## PINN for Cahn-Hilliard Equation (Example 4.1)

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.autograd import grad

import mlflow
import mlflow.pytorch

from tqdm.notebook import tqdm

import imageio.v2 as imageio
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

import os

### 1-D Cahn-Hilliard Equation

$u_t = 6uu_x^2 + (3u^2-1)u_{xx}-\epsilon^2u_{xxxx}$

$ x \in [0, 2\pi], t \ge 0$

$ u(0, x) = 0.5\cos(x)$

$ ϵ = \sqrt{0.03}$

In [2]:
ϵ = np.sqrt(0.03)

In [3]:
physical_params = {
    'ϵ': ϵ,
}

In [4]:
def initial_cond(t, x):
    return 0.5*torch.cos(x)

In [5]:
class PINN(nn.Module):
    def __init__(self, pars: dict):
        super().__init__()
        self.pars = pars
        
        self.modules = [nn.BatchNorm1d(2), nn.Linear(2, self.pars['width'])] # nn.LayerNorm(2)
        for i in range(self.pars['layers'] - 1):
            # self.modules.append(nn.LayerNorm(self.pars['width']))
            self.modules.append(nn.Tanh())
            self.modules.append(nn.Linear(self.pars['width'], self.pars['width']))
            
        self.modules.append(nn.Tanh())
        self.modules.append(nn.Linear(self.pars['width'], 1))
        
        self.model = nn.Sequential(*self.modules)
        self.model.to(self.pars['device'])
        
        self.optimizer = torch.optim.Adam(params=self.model.parameters(), lr=self.pars['lr'])
        self.num_params = sum([len(params) for params in [p for p in self.model.parameters()]])
        
        self.epoch = 0
        
        t = np.linspace(self.pars['t_min'], self.pars['t_max'], 100)
        x = np.linspace(self.pars['x_min'], self.pars['x_max'], 100)

        self.eval_t, self.eval_x = np.meshgrid(t, x)
        self.eval_t = torch.Tensor(self.eval_t).reshape(-1, 1).to(self.pars['device'])
        self.eval_x = torch.Tensor(self.eval_x).reshape(-1, 1).to(self.pars['device'])
        
        self.eval_t.requires_grad_()
        self.eval_x.requires_grad_()
        
        eval_t_interior, eval_x_interior = np.meshgrid(t[1:-1], x[1:-1])
        eval_t_interior = torch.Tensor(eval_t_interior).reshape(-1, 1).to(self.pars['device'])
        eval_x_interior = torch.Tensor(eval_x_interior).reshape(-1, 1).to(self.pars['device'])
        
        self.eval_X_interior = torch.hstack((eval_t_interior, eval_x_interior))
        self.eval_X_interior.requires_grad_()
        
        eval_t_initial = torch.zeros_like(self.eval_x)
        self.eval_X_initial = torch.hstack((eval_t_initial, self.eval_x))
        self.eval_X_initial.requires_grad_()
        
        eval_t_boundary = torch.Tensor(np.vstack([t, t])).reshape(-1, 1).to(self.pars['device'])
        eval_x_boundary = torch.Tensor(np.vstack([x[0] * np.ones_like(t), x[-1] * np.ones_like(t)])).reshape(-1, 1).to(self.pars['device'])
        
        self.eval_X_boundary = torch.hstack((eval_t_boundary, eval_x_boundary))
        self.eval_X_boundary.requires_grad_()
        
        self.plot_t = self.eval_t.view(100, 100).detach().cpu().numpy()
        self.plot_x = self.eval_x.view(100, 100).detach().cpu().numpy()
        
        self.plot_files = {}
        
        if 'plot_vars_list' in self.pars:
            for varname in self.pars['plot_vars_list']:
                self.plot_files[varname] = []
        
    def __call__(self, X):
        return self.model(X)
        
    def sample_interior_points(self):
        t = torch.empty((self.pars['interior_batch_size'], 1), device=self.pars['device']).uniform_(self.pars['t_min'], self.pars['t_max'])
        x = torch.empty((self.pars['interior_batch_size'], 1), device=self.pars['device']).uniform_(self.pars['x_min'], self.pars['x_max'])
        X_interior = torch.cat((t, x), 1)
        X_interior.requires_grad_()
        
        return X_interior
    
    def sample_initial_points(self):
        t = torch.zeros(self.pars['initial_batch_size'], 1, device=self.pars['device'])
        x = torch.empty((self.pars['initial_batch_size'], 1), device=self.pars['device']).uniform_(self.pars['x_min'], self.pars['x_max'])
        X_initial = torch.cat((t, x), 1)
        X_initial.requires_grad_()
        
        return X_initial
    
    def sample_boundary_points(self):
        options = torch.tensor([self.pars['x_min'], self.pars['x_max']], device=self.pars['device'])
        
        t = torch.empty((self.pars['boundary_batch_size'], 1), device=self.pars['device']).uniform_(self.pars['t_min'], self.pars['t_max'])
        x = options[torch.randint(0, 2, (self.pars['boundary_batch_size'], 1), device=self.pars['device'])]
        X_boundary = torch.cat((t, x), 1)
        X_boundary.requires_grad_()
        
        return X_boundary
    
    def forward(self, X_interior, X_initial, X_boundary):
        # X shape: (batch_size, 2), where 2nd dimension is [t, x]
        # u shape: (batch_size, 1)
        
        t_interior = X_interior[:, 0].reshape(-1, 1)
        x_interior = X_interior[:, 1].reshape(-1, 1)
        t_initial = X_initial[:, 0].reshape(-1, 1)
        x_initial = X_initial[:, 1].reshape(-1, 1)
        t_boundary = X_boundary[:, 0].reshape(-1, 1)
        x_boundary = X_boundary[:, 1].reshape(-1, 1)
        
        # forward pass
        u_interior = self.model(torch.hstack((t_interior, x_interior)))
        u_initial = self.model(torch.hstack((t_initial, x_initial)))
        u_boundary = self.model(torch.hstack((t_boundary, x_boundary)))
        
        u_t_interior = grad(u_interior, t_interior, grad_outputs=torch.ones_like(u_interior), retain_graph=True, create_graph=True)[0]
        u_x_interior = grad(u_interior, x_interior, grad_outputs=torch.ones_like(u_interior), retain_graph=True, create_graph=True)[0]
        
        u_xx_interior = grad(u_x_interior, x_interior, grad_outputs=torch.ones_like(u_x_interior), retain_graph=True, create_graph=True)[0]
        u_xxx_interior = grad(u_xx_interior, x_interior, grad_outputs=torch.ones_like(u_xx_interior), retain_graph=True, create_graph=True)[0]
        u_xxxx_interior = grad(u_xxx_interior, x_interior, grad_outputs=torch.ones_like(u_xxx_interior), retain_graph=True, create_graph=True)[0]
        
        u_x_boundary = grad(u_boundary, x_boundary, grad_outputs=torch.ones_like(u_boundary), retain_graph=True, create_graph=True)[0]
        
        pde_loss = ((u_t_interior - 6*u_interior*u_x_interior**2 -(3*u_interior**2 - 1)*u_xx_interior + ϵ**2*u_xxxx_interior)**2).mean()
        
        mse = nn.MSELoss()
        initial_loss = mse(u_initial, initial_cond(t_initial, x_initial))
        
        boundary_loss = (u_x_boundary**2).mean()
        
        total_loss = pde_loss + initial_loss + boundary_loss
        
        return total_loss, pde_loss, initial_loss, boundary_loss
    
    def train(self):
        cwd = os.getcwd()
        plot_dirs = os.path.join(cwd, f'plots/{self.pars["experiment_name"]}')

        if not os.path.isdir(plot_dirs):
            os.makedirs(plot_dirs)
        
        mlflow.set_experiment(self.pars['experiment_name'])
        mlflow.start_run()
        
        mlflow.log_param("physical_params", physical_params)
        mlflow.log_param("model_params", self.pars)
        
        for epoch in tqdm(range(self.pars['epochs']), position=0, leave=True, desc='Training...'): 
            self.epoch = epoch
            
            # eval
            if epoch % self.pars['eval_interval'] == 0 or epoch == self.pars['epochs'] - 1:
                
                total_loss, pde_loss, initial_loss, boundary_loss = self.forward(self.eval_X_interior, self.eval_X_initial, self.eval_X_boundary)
                
                print()
                print(f'Epoch: {self.epoch}, Loss: {total_loss.item():,.4e}')
                print(f"pde_loss: {pde_loss.item():.4e}, initial_loss: {initial_loss.item():.4e}, boundary_loss: {boundary_loss:.4e}")

                mlflow.log_metric("total_loss", total_loss.item(), step=self.epoch)
                mlflow.log_metric("pde_loss", pde_loss.item(), step=self.epoch)
                mlflow.log_metric("initial_loss", initial_loss.item(), step=self.epoch)
                mlflow.log_metric("boundary_loss", boundary_loss, step=self.epoch)
                
                mlflow.pytorch.log_model(self.model, f"{self.pars['experiment_name']}_model_epoch_{self.epoch}")
                
                if self.pars['plot_training_outputs']:
                    self.plot_outputs()
            
            # training step
            self.optimizer.zero_grad()
            
            if epoch % self.pars['sample_interval'] == 0:
                X_interior = self.sample_interior_points()
                X_initial = self.sample_initial_points()
                X_boundary = self.sample_boundary_points()
            total_loss, pde_loss, initial_loss, boundary_loss = self.forward(X_interior, X_initial, X_boundary)
            total_loss.backward()
            self.optimizer.step()
            
                
            if total_loss.isnan():
                print(f'Epoch: {self.epoch}, Loss: {total_loss.item():,.4e}')
                print(f"pde_loss: {pde_loss.item():.4e}, initial_loss: {initial_loss.item():.4e}, boundary_loss: {boundary_loss:.4e}")
                print("loss is NaN, stopping training...")
                break
            
        mlflow.pytorch.log_model(self.model, f"{self.pars['experiment_name']}_model_final")
        
        self.save_gif()
        
        mlflow.end_run()
    
    def plot_outputs(self):
        u = self(torch.hstack((self.eval_t, self.eval_x)))
        
        u_t = grad(u, self.eval_t, grad_outputs=torch.ones_like(u), retain_graph=True)[0]
        u_x = grad(u, self.eval_x, grad_outputs=torch.ones_like(u), retain_graph=True, create_graph=True)[0]
        u_xx = grad(u_x, self.eval_x, grad_outputs=torch.ones_like(u_x), retain_graph=True, create_graph=True)[0]
        u_xxx = grad(u_xx, self.eval_x, grad_outputs=torch.ones_like(u_xx), retain_graph=True, create_graph=True)[0]
        u_xxxx = grad(u_xxx, self.eval_x, grad_outputs=torch.ones_like(u_xxx), retain_graph=True)[0]
        
        u = u.view(100, 100).detach().cpu().numpy()
        u_t = u_t.view(100, 100).detach().cpu().numpy()
        u_x = u_x.view(100, 100).detach().cpu().numpy()
        u_xx = u_xx.view(100, 100).detach().cpu().numpy()
        u_xxx = u_xxx.view(100, 100).detach().cpu().numpy()
        u_xxxx = u_xxxx.view(100, 100).detach().cpu().numpy()
        
        if 'u' in self.pars['plot_vars_list']:
            self.plot_var(u, 'u')
        
        if 'u_x' in self.pars['plot_vars_list']:
            self.plot_var(u_x, 'u_x')
        
        if 'u_t' in self.pars['plot_vars_list']:
            self.plot_var(u_t, 'u_t')
        
        if 'u_xx' in self.pars['plot_vars_list']:
            self.plot_var(u_xx, 'u_xx')
        
        if 'u_xxx' in self.pars['plot_vars_list']:
            self.plot_var(u_xxx, 'u_xxx')
        
        if 'u_xxxx' in self.pars['plot_vars_list']:
            self.plot_var(u_xxxx, 'u_xxxx')
        
    def plot_var(self, var, varname):
        fig = plt.figure()
        ax = fig.add_subplot(111, projection='3d')
        ax.plot_surface(self.plot_t, self.plot_x, var, cmap='viridis')
        ax.set_xlabel('t')
        ax.set_ylabel('x')
        ax.set_zlabel(varname)
        plt.title(f'Model output {varname}, epoch {self.epoch}')
        filename = f"plots/{self.pars['experiment_name']}/plot_{varname}_epoch_{self.epoch}.png"
        self.plot_files[varname].append(filename)
        fig.savefig(filename)
        mlflow.log_artifact(filename)
        plt.close()
        
    def save_gif(self):
        for varname in self.pars['plot_vars_list']:
            gif_filename = f"plots/{self.pars['experiment_name']}/plot_{varname}.gif"
            with imageio.get_writer(gif_filename, mode='I') as writer:
                for filename in self.plot_files[varname]:
                    image = imageio.imread(filename)
                    writer.append_data(image)
                
            mlflow.log_artifact(gif_filename)

In [6]:
pars = {
    'experiment_name': 'cahn_hilliard',
    'layers': 4,
    'width': 128,
    'lr': 1e-4,
    'epochs': 10000,
    'eval_interval': 500,
    'interior_batch_size': 2048,
    'initial_batch_size': 2048,
    'boundary_batch_size': 2048,
    'x_min': 0.0,
    'x_max': 2*np.pi,
    't_min': 0.0,
    't_max': 15,
    'device': 'cuda',
    'plot_training_outputs': True,
    'plot_vars_list': ['u', 'u_t', 'u_x', 'u_xx', 'u_xxx', 'u_xxxx'],
    'sample_interval': np.inf
}
pinn = PINN(pars)


In [7]:
pinn.train()

Training...:   0%|          | 0/10000 [00:00<?, ?it/s]

/home/david/anaconda3/envs/pdenn/lib/python3.12/site-packages/torch/autograd/graph.py:768: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass



Epoch: 0, Loss: 1.2814e-01
pde_loss: 6.8261e-06, initial_loss: 1.2811e-01, boundary_loss: 1.9340e-05


2025/01/21 11:35:00 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.



Epoch: 500, Loss: 3.2576e-03
pde_loss: 1.8645e-03, initial_loss: 1.0069e-03, boundary_loss: 3.8623e-04


...

2025/01/21 11:41:53 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.



Epoch: 9500, Loss: 8.2180e-04
pde_loss: 5.5248e-04, initial_loss: 2.6026e-04, boundary_loss: 9.0530e-06


2025/01/21 11:42:16 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.



Epoch: 9999, Loss: 7.9263e-04
pde_loss: 5.5132e-04, initial_loss: 2.3323e-04, boundary_loss: 8.0787e-06


2025/01/21 11:42:40 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.
2025/01/21 11:42:42 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.
